In [1]:
# ============================================================
# STEP 1: Install and import required libraries
# ============================================================

# Install Hugging Face datasets library for loading the dataset
# Install sentence-transformers for later clustering/semantic filtering
# Install openai for persona rewriting and survey generation later
!pip install -q datasets pandas pyarrow sentence-transformers scikit-learn openai

# Core data libraries
import pandas as pd
import numpy as np
import json
import os
from pprint import pprint

# Hugging Face dataset loader
from datasets import load_dataset

# For later clustering
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans

print("✅ Step 1 complete: Libraries installed and imported.")

✅ Step 1 complete: Libraries installed and imported.


In [2]:
# ============================================================
# STEP 2: Load a manageable sample of the dataset
# ============================================================

# Load only first 20,000 rows (fast + safe)
ds = load_dataset(
    "nvidia/Nemotron-Personas-USA",
    split="train[:20000]"
)

# Convert to pandas DataFrame for easier processing
df = ds.to_pandas()

# Basic sanity checks
print("Shape of dataset:", df.shape)

# Show column names
print("\nColumns available:")
print(df.columns.tolist())

# Show 2 sample rows (important to understand structure)
print("\nSample rows:")
display(df.head(2))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00011.parquet:   0%|          | 0.00/244M [00:00<?, ?B/s]

data/train-00001-of-00011.parquet:   0%|          | 0.00/244M [00:00<?, ?B/s]

data/train-00002-of-00011.parquet:   0%|          | 0.00/244M [00:00<?, ?B/s]

...

data/train-00009-of-00011.parquet:   0%|          | 0.00/244M [00:00<?, ?B/s]

data/train-00010-of-00011.parquet:   0%|          | 0.00/244M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000000 [00:00<?, ? examples/s]

Shape of dataset: (20000, 23)

Columns available:
['uuid', 'professional_persona', 'sports_persona', 'arts_persona', 'travel_persona', 'culinary_persona', 'persona', 'cultural_background', 'skills_and_expertise', 'skills_and_expertise_list', 'hobbies_and_interests', 'hobbies_and_interests_list', 'career_goals_and_ambitions', 'sex', 'age', 'marital_status', 'education_level', 'bachelors_field', 'occupation', 'city', 'state', 'zipcode', 'country']

Sample rows:


,uuid,professional_persona,sports_persona,arts_persona,travel_persona,culinary_persona,persona,cultural_background,skills_and_expertise,skills_and_expertise_list,...,sex,age,marital_status,education_level,bachelors_field,occupation,city,state,zipcode,country
0,e7c0574639a244c8972c92aab9501035,Mary Alberti is a front‑line food service spec...,Mary Alberti fuels their fitness routine by cl...,Mary Alberti finds creative inspiration in the...,Mary Alberti prefers meticulously planned week...,Mary Alberti showcases an intermediate culinar...,"Mary Alberti is a routine‑obsessed, bullet‑jou...",Mary is a second‑generation Italian‑American r...,Mary has become highly proficient in front‑lin...,"['POS system operation', 'Cash handling', 'Foo...",...,Female,28,never_married,high_school,,fast_food_or_counter_worker,Madison,WI,53717,USA
1,50f90a6f17de473f9ca15f00afdedf7a,"Alicia Gonzalez, a research scientist speciali...","Alicia Gonzalez, an enthusiastic hiker who spe...","Alicia Gonzalez, an abstract painter and gener...","Alicia Gonzalez, who mixes meticulous research...","Alicia Gonzalez, an intermediate‑to‑advanced h...",Alicia Gonzalez fuses scientific precision wit...,Alicia was raised in a bilingual Colombian hou...,Alicia is a computer and information research ...,"['Machine Learning', 'Deep Learning', 'Natural...",...,Female,29,divorced,bachelors,stem,computer_or_information_research_scientist,Chico,CA,95928,USA


In [3]:
# ============================================================
# STEP 3: Inspect and clean important columns
# ============================================================

# Check missing values in key columns
key_cols = [
    "persona",
    "professional_persona",
    "occupation",
    "age",
    "sex",
    "marital_status",
    "education_level",
    "city",
    "state",
    "country",
    "skills_and_expertise",
    "hobbies_and_interests",
    "career_goals_and_ambitions",
    "cultural_background"
]

print("Missing values check:\n")
print(df[key_cols].isnull().sum())

# Fill missing values with empty string (important for text processing)
for col in key_cols:
    df[col] = df[col].fillna("").astype(str)

# Convert age to numeric (sometimes comes as string)
df["age"] = pd.to_numeric(df["age"], errors="coerce")

# Drop rows where age is missing (we need it for filtering)
df = df.dropna(subset=["age"])

# Reset index after cleaning
df = df.reset_index(drop=True)

print("\nShape after cleaning:", df.shape)

# ------------------------------------------------------------
# Create a combined text field (VERY IMPORTANT)
# This helps us search/filter across multiple columns at once
# ------------------------------------------------------------

df["combined_text"] = (
    df["persona"] + " " +
    df["professional_persona"] + " " +
    df["occupation"] + " " +
    df["hobbies_and_interests"] + " " +
    df["career_goals_and_ambitions"] + " " +
    df["cultural_background"]
).str.lower()

# Show a sample combined text
print("\nSample combined text:")
print(df["combined_text"].iloc[0][:500])

Missing values check:

persona                       0
professional_persona          0
occupation                    0
age                           0
sex                           0
marital_status                0
education_level               0
city                          0
state                         0
country                       0
skills_and_expertise          0
hobbies_and_interests         0
career_goals_and_ambitions    0
cultural_background           0
dtype: int64

Shape after cleaning: (20000, 23)

Sample combined text:
mary alberti is a routine‑obsessed, bullet‑journal aficionado who balances disciplined work ambition with a competitive edge, occasional craft‑brew indulgence, and a habit of double‑checking every receipt for hidden costs. mary alberti is a front‑line food service specialist whose razor‑sharp cash handling, inventory tracking, and pos mastery combine with a disciplined, routine‑driven work ethic, enabling them to calmly resolve high‑pressure customer iss

In [4]:
# ============================================================
# STEP 4: Filter likely parents of children age 4–10
# ============================================================

# ---------------------------
# 1. Hard filter: age range
# ---------------------------
# Parents of kids 4–10 are typically 25–50

df_parents = df[
    (df["age"] >= 25) & (df["age"] <= 50)
].copy()

print("After age filter:", df_parents.shape)


# ---------------------------
# 2. Define keyword groups
# ---------------------------

PARENT_KEYWORDS = [
    "parent", "mother", "mom", "mum", "father", "dad",
    "raising children", "raising kids", "kids", "children",
    "son", "daughter"
]

FAMILY_KEYWORDS = [
    "family", "household", "married life",
    "weekend with family", "family activities"
]

KID_AGE_4_10_HINTS = [
    "elementary school", "school pickup", "school drop",
    "after-school", "homework", "playdate",
    "birthday party", "kids activities"
]


# ---------------------------
# 3. Scoring function
# ---------------------------

def compute_parent_score(text, marital_status):
    score = 0

    # Strong indicators
    for kw in PARENT_KEYWORDS:
        if kw in text:
            score += 3

    # Medium indicators
    for kw in FAMILY_KEYWORDS:
        if kw in text:
            score += 1

    # Age-specific hints
    for kw in KID_AGE_4_10_HINTS:
        if kw in text:
            score += 2

    # Weak signal: marital status
    if marital_status in ["married", "divorced", "separated"]:
        score += 1

    return score


# Apply scoring
df_parents["parent_score"] = df_parents.apply(
    lambda row: compute_parent_score(row["combined_text"], row["marital_status"]),
    axis=1
)


# ---------------------------
# 4. Filter strong candidates
# ---------------------------

df_target = df_parents[df_parents["parent_score"] >= 4].copy()

print("After parent scoring filter:", df_target.shape)


# ---------------------------
# 5. Optional: check top results
# ---------------------------

df_target = df_target.sort_values("parent_score", ascending=False)

print("\nTop candidates preview:")
display(df_target[[
    "age", "marital_status", "occupation", "parent_score"
]].head(10))


# Show one example full text (important for sanity check)
print("\nSample persona text:\n")
print(df_target["combined_text"].iloc[0][:600])

After age filter: (6824, 24)
After parent scoring filter: (5553, 25)

Top candidates preview:


,age,marital_status,occupation,parent_score
19615,31,divorced,chemical_engineer,16
16093,25,never_married,preschool_or_kindergarten_teacher,16
13352,39,married_present,construction_or_building_inspector,15
4024,33,divorced,elementary_or_middle_school_teacher,15
13898,25,never_married,customer_service_representative,15
8564,42,divorced,no_occupation,14
10708,41,married_present,manager,14
12581,46,divorced,maid_or_housekeeping_cleaner,14
14665,43,divorced,preschool_or_kindergarten_teacher,14
16081,36,divorced,not_in_workforce,14



Sample persona text:

emma jackson merges a scientifically disciplined mind with an artistic soul, navigating life with organized ambition, competitive edge, and a love for collaborative, sensory‑rich experiences, even if occasional skepticism and busy‑life slip‑ups keep them grounded. emma jackson is a chemical engineer who channels rigorous process design expertise into sustainable pigment r&d, leveraging their meticulous project management, persuasive technical writing, and competitive drive to lead cross‑functional teams while translating complex chemistry into vibrant visual narratives for art‑focused markets.


In [5]:
# ============================================================
# STEP 5: Create financial behavior segments
# ============================================================

# ---------------------------
# 1. Define keyword signals
# ---------------------------

FINANCE_SIGNALS = {
    "security": [
        "saving", "safe", "security", "stable", "protect",
        "risk-averse", "guarantee"
    ],
    "growth": [
        "invest", "portfolio", "stocks", "wealth",
        "long-term", "returns", "financial growth"
    ],
    "busy": [
        "busy", "schedule", "time", "work-life balance",
        "overwhelmed", "no time"
    ],
    "budget": [
        "budget", "afford", "expenses", "cost",
        "saving money", "financial constraints"
    ],
    "values": [
        "teach", "learning", "kids development",
        "habits", "education", "values"
    ],
    "skeptical": [
        "concern", "uncertain", "trust", "confused",
        "risk", "fear", "complex"
    ]
}


# ---------------------------
# 2. Classification function
# ---------------------------

def classify_financial_segment(text):
    scores = {k: 0 for k in FINANCE_SIGNALS}

    for segment, keywords in FINANCE_SIGNALS.items():
        for kw in keywords:
            if kw in text:
                scores[segment] += 1

    # Pick the highest scoring segment
    best_segment = max(scores, key=scores.get)

    # If all scores are zero → default
    if scores[best_segment] == 0:
        return "general"

    return best_segment


# Apply classification
df_target["finance_segment_raw"] = df_target["combined_text"].apply(
    classify_financial_segment
)


# ---------------------------
# 3. Map to business personas
# ---------------------------

SEGMENT_MAP = {
    "security": "Security-First Parent",
    "growth": "Growth-Oriented Parent",
    "busy": "Busy Convenience Parent",
    "budget": "Budget-Conscious Parent",
    "values": "Values-Driven Parent",
    "skeptical": "Skeptical Parent",
    "general": "General Parent"
}

df_target["finance_segment"] = df_target["finance_segment_raw"].map(SEGMENT_MAP)


# ---------------------------
# 4. Inspect distribution
# ---------------------------

print("Segment distribution:\n")
print(df_target["finance_segment"].value_counts())


# ---------------------------
# 5. Preview sample personas
# ---------------------------

print("\nSample personas by segment:\n")

for seg in df_target["finance_segment"].unique():
    sample = df_target[df_target["finance_segment"] == seg].iloc[0]

    print(f"\n=== {seg} ===")
    print(f"Age: {sample['age']} | Occupation: {sample['occupation']}")
    print(sample["combined_text"][:300])

Segment distribution:

finance_segment
Values-Driven Parent       2119
Busy Convenience Parent    1894
Security-First Parent       964
Growth-Oriented Parent      265
Budget-Conscious Parent     190
General Parent               68
Skeptical Parent             53
Name: count, dtype: int64

Sample personas by segment:


=== Busy Convenience Parent ===
Age: 31 | Occupation: chemical_engineer
emma jackson merges a scientifically disciplined mind with an artistic soul, navigating life with organized ambition, competitive edge, and a love for collaborative, sensory‑rich experiences, even if occasional skepticism and busy‑life slip‑ups keep them grounded. emma jackson is a chemical engineer

=== Values-Driven Parent ===
Age: 25 | Occupation: preschool_or_kindergarten_teacher
vivian mallari, a 25‑year‑old early‑childhood educator, balances a love for hands‑on stem, community‑driven rituals, and a modest budget, while occasionally over‑knitting scarves and forgetting to floss. a 25‑year‑old ear

In [6]:
# ============================================================
# STEP 6: Build a balanced persona panel (10–12 personas)
# ============================================================

# 1) Deduplicate by core persona text (avoid near repeats)
df_unique = df_target.drop_duplicates(subset=["persona"]).copy()

# 2) Sort by strongest parent signals first
df_unique = df_unique.sort_values("parent_score", ascending=False)

# 3) Decide how many personas per segment (aim total ~10–12)
# You can tweak this if some segments are very small
PER_SEGMENT = 2

# 4) Select top personas per segment
panel_rows = []
segments = df_unique["finance_segment"].dropna().unique().tolist()

for seg in segments:
    seg_df = df_unique[df_unique["finance_segment"] == seg].copy()

    # Take top N per segment
    top_n = seg_df.head(PER_SEGMENT)

    panel_rows.append(top_n)

# Combine
panel_df = pd.concat(panel_rows, axis=0).reset_index(drop=True)

# 5) If we have too many (>12), trim globally by parent_score
if len(panel_df) > 12:
    panel_df = panel_df.sort_values("parent_score", ascending=False).head(12)

# 6) Create a simple readable label
def make_label(row):
    return f"{row['finance_segment']} | {int(row['age'])} | {row['occupation']} | {row['state']}"

panel_df["panel_label"] = panel_df.apply(make_label, axis=1)

# 7) Quick view
print("Final panel size:", len(panel_df))
display(panel_df[[
    "panel_label",
    "finance_segment",
    "age",
    "occupation",
    "state",
    "parent_score"
]])

Final panel size: 12


,panel_label,finance_segment,age,occupation,state,parent_score
0,Busy Convenience Parent | 31 | chemical_engine...,Busy Convenience Parent,31,chemical_engineer,FL,16
2,Values-Driven Parent | 25 | preschool_or_kinde...,Values-Driven Parent,25,preschool_or_kindergarten_teacher,NY,16
4,Growth-Oriented Parent | 25 | customer_service...,Growth-Oriented Parent,25,customer_service_representative,DC,15
3,Values-Driven Parent | 33 | elementary_or_midd...,Values-Driven Parent,33,elementary_or_middle_school_teacher,NY,15
1,Busy Convenience Parent | 46 | not_in_workforc...,Busy Convenience Parent,46,not_in_workforce,FL,14
5,Growth-Oriented Parent | 42 | no_occupation | TX,Growth-Oriented Parent,42,no_occupation,TX,14
6,General Parent | 48 | cabinetmaker_or_bench_ca...,General Parent,48,cabinetmaker_or_bench_carpenter,WV,14
8,Security-First Parent | 37 | painting_worker | MA,Security-First Parent,37,painting_worker,MA,14
9,Security-First Parent | 48 | heavy_vehicle_or_...,Security-First Parent,48,heavy_vehicle_or_mobile_equipment_service_tech...,NY,14
10,Skeptical Parent | 38 | animal_caretaker | TX,Skeptical Parent,38,animal_caretaker,TX,14


In [33]:
# ============================================================
# STEP 7: Robust persona generation with JSON mode
# ============================================================

import json
import os
from getpass import getpass
from openai import OpenAI

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

financial_product = """
FlexiSave Kids Plan is a new financial product for parents of children aged 4–10.

It combines:
1. Goal-based savings for children's future needs such as education, activities, and enrichment.
2. Optional low-risk micro-investing.
3. Monthly automatic contributions.
4. Simple money-learning nudges for children.
5. Parent dashboard showing progress toward goals.
"""

def create_clean_persona(row):
    prompt = f"""
Create one UX research persona for testing this financial product:

{financial_product}

Raw synthetic persona data:
Age: {row['age']}
Sex: {row['sex']}
Marital status: {row['marital_status']}
Occupation: {row['occupation']}
Education: {row['education_level']}
Location: {row['city']}, {row['state']}, USA
Financial segment: {row['finance_segment']}
Core persona: {row['persona']}
Professional persona: {row['professional_persona']}
Skills: {row['skills_and_expertise']}
Hobbies: {row['hobbies_and_interests']}
Career goals: {row['career_goals_and_ambitions']}
Cultural background: {row['cultural_background']}

Rules:
- Return JSON only.
- Do not invent unsupported facts.
- Parent status is inferred, not confirmed.
- Make it useful for survey design.

Return this JSON structure:
{{
  "name": "realistic first name only",
  "persona_title": "short title",
  "segment": "{row['finance_segment']}",
  "snapshot": "2-3 sentence profile",
  "financial_mindset": "how this person thinks about money",
  "parenting_context": "how being a likely parent of a 4-10 year old may affect decisions",
  "goals": ["goal 1", "goal 2", "goal 3"],
  "pain_points": ["pain point 1", "pain point 2", "pain point 3"],
  "purchase_triggers": ["trigger 1", "trigger 2", "trigger 3"],
  "objections": ["objection 1", "objection 2", "objection 3"],
  "survey_angle": "what this persona is useful for testing",
  "quote": "short realistic quote"
}}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        response_format={"type": "json_object"},
        messages=[
            {
                "role": "system",
                "content": "You create realistic UX research personas. Return valid JSON only."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.4
    )

    return json.loads(response.choices[0].message.content)


clean_personas = []

for i, row in panel_df.iterrows():
    print(f"Creating persona {i+1}/{len(panel_df)}...")

    try:
        persona = create_clean_persona(row)
        persona["source_uuid"] = row["uuid"]
        persona["source_panel_label"] = row["panel_label"]
        clean_personas.append(persona)

    except Exception as e:
        print(f"❌ Failed row {i}: {e}")

print("\nPersonas created:", len(clean_personas))

if len(clean_personas) == 0:
    raise ValueError("No personas created. Check panel_df and API access.")

with open("clean_financial_parent_personas.json", "w") as f:
    json.dump(clean_personas, f, indent=2)

print("✅ Saved clean_financial_parent_personas.json")

#for p in clean_personas[:3]:
#    print("\n", p["name"], "-", p["persona_title"])
#    print(p["snapshot"])

# ------------------------------------------------------------
# Display clean personas
# ------------------------------------------------------------

for i, p in enumerate(clean_personas, start=1):
    print("\n" + "="*70)
    print(f"PERSONA {i}: {p['name']} — {p['persona_title']}")
    print("="*70)
    print("Segment:", p["segment"])
    print("Snapshot:", p["snapshot"])
    print("Financial mindset:", p["financial_mindset"])
    print("Survey angle:", p["survey_angle"])
    print("Quote:", p["quote"])

Creating persona 1/12...
Creating persona 3/12...
Creating persona 5/12...
Creating persona 4/12...
Creating persona 2/12...
Creating persona 6/12...
Creating persona 7/12...
Creating persona 9/12...
Creating persona 10/12...
Creating persona 11/12...
Creating persona 14/12...
Creating persona 13/12...

Personas created: 12
✅ Saved clean_financial_parent_personas.json

PERSONA 1: Emma — Ambitious Parent
Segment: Busy Convenience Parent
Snapshot: Emma is a 31-year-old chemical engineer who balances a demanding career with her passion for art and community. As a likely parent of a young child, she seeks efficient and meaningful ways to save for her child's future while nurturing their creativity and learning.
Financial mindset: Emma views money as a tool for empowerment and opportunity, prioritizing investments that align with her values of sustainability and education. She is cautious yet open to innovative financial solutions that can help her achieve her goals.
Survey angle: This pers

In [19]:
# ============================================================
# STEP 8: Regenerate a master survey
# ============================================================

import json

def create_master_survey_fixed(clean_personas, financial_product):
    persona_summary = []

    for p in clean_personas:
        persona_summary.append({
            "name": p.get("name"),
            "segment": p.get("segment"),
            "snapshot": p.get("snapshot"),
            "financial_mindset": p.get("financial_mindset"),
            "pain_points": p.get("pain_points"),
            "objections": p.get("objections"),
            "survey_angle": p.get("survey_angle")
        })

    prompt = f"""
You are a senior UX researcher.

Create a survey to test this financial product:

{financial_product}

The survey is for parents evaluating this product.

Use these personas as research inputs:
{json.dumps(persona_summary, indent=2)}

Return VALID JSON only.

Important:
- You MUST create exactly 5 sections.
- Each section MUST contain exactly 3 questions.
- Total questions MUST be 15.
- Do NOT return an empty list.
- Each question must be useful for testing product-market fit.

Use this exact JSON structure:

{{
  "sections": [
    {{
      "section_name": "Section name",
      "questions": [
        {{
          "question": "Question text",
          "type": "multiple_choice",
          "options": ["Option A", "Option B", "Option C", "Option D"],
          "why_it_matters": "Reason"
        }}
      ]
    }}
  ]
}}

Allowed question types:
- multiple_choice
- likert
- open
"""

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {
                "role": "system",
                "content": "You create complete survey questionnaires and return valid non-empty JSON only."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.3
    )

    survey = json.loads(response.choices[0].message.content)

    # Validation
    if "sections" not in survey or len(survey["sections"]) == 0:
        raise ValueError("Survey generation failed: sections list is empty.")

    total_questions = sum(len(section.get("questions", [])) for section in survey["sections"])

    if total_questions == 0:
        raise ValueError("Survey generation failed: total questions is zero.")

    print(f"✅ Survey generated with {len(survey['sections'])} sections and {total_questions} questions.")

    return survey


master_survey = create_master_survey_fixed(clean_personas, financial_product)

# Display survey
for section in master_survey["sections"]:
    print("\n" + "="*70)
    print(section["section_name"])
    print("="*70)

    for q in section["questions"]:
        print("\nQ:", q["question"])
        print("Type:", q["type"])
        print("Why:", q["why_it_matters"])

# Save fixed survey
with open("financial_product_survey.json", "w") as f:
    json.dump(master_survey, f, indent=2)

print("\n✅ Saved fixed survey to financial_product_survey.json")

✅ Survey generated with 5 sections and 15 questions.

Overall Interest and Appeal

Q: How interested are you in using a financial product like FlexiSave Kids Plan for your child?
Type: likert
Why: Measures initial appeal and potential adoption likelihood.

Q: Which feature of the FlexiSave Kids Plan appeals to you the most?
Type: multiple_choice
Why: Identifies the most valued product features to prioritize.

Q: How likely are you to recommend FlexiSave Kids Plan to other parents?
Type: likert
Why: Assesses potential for word-of-mouth growth and satisfaction.

Savings and Investment Features

Q: How important is the option of low-risk micro-investing in your decision to use this product?
Type: likert
Why: Evaluates demand for investment options within a kids savings product.

Q: Do you find monthly automatic contributions helpful for building savings consistently?
Type: multiple_choice
Why: Tests acceptance and perceived usefulness of automated savings.

Q: What concerns, if any, do yo

In [26]:
# ============================================================
# STEP 9: Simulate survey responses from each persona
# ============================================================

# ------------------------------------------------------------
# 1. Helper: flatten survey into a simple question list
# ------------------------------------------------------------

def extract_questions(master_survey):
    questions = []
    for section in master_survey["sections"]:
        for q in section["questions"]:
            questions.append({
                "question": q["question"],
                "type": q["type"]
            })
    return questions

survey_questions = extract_questions(master_survey)

print(f"Total questions: {len(survey_questions)}")


# ------------------------------------------------------------
# 2. Function to simulate persona answering survey
# ------------------------------------------------------------

def simulate_response(persona, survey_questions):
    prompt = f"""
You are roleplaying this persona:

Name: {persona['name']}
Segment: {persona['segment']}
Snapshot: {persona['snapshot']}
Financial mindset: {persona['financial_mindset']}
Pain points: {persona['pain_points']}
Objections: {persona['objections']}

You are answering a survey for this product:

{financial_product}

Instructions:
- Answer realistically and consistently
- Be honest (do NOT try to please the product)
- Reflect your concerns and constraints
- Keep answers concise but meaningful

Return JSON:

{{
  "persona_name": "{persona['name']}",
  "responses": [
    {{
      "question": "...",
      "answer": "...",
      "reasoning": "why you answered this way"
    }}
  ]
}}

Survey:
{json.dumps(survey_questions, indent=2)}
"""

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": "You simulate realistic user behavior."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.6
    )

    return json.loads(response.choices[0].message.content)


# ------------------------------------------------------------
# 3. Run simulation for all personas
# ------------------------------------------------------------

all_responses = []

for i, persona in enumerate(clean_personas):
    print(f"Simulating responses for {persona['name']} ({i+1}/{len(clean_personas)})...")

    result = simulate_response(persona, survey_questions)
    all_responses.append(result)

print("\n✅ Simulation complete.")


# ------------------------------------------------------------
# 4. Display sample output
# ------------------------------------------------------------

for r in all_responses[:2]:
    print("\n" + "="*70)
    print(f"PERSONA: {r['persona_name']}")
    print("="*70)

    for ans in r["responses"][:5]:
        print(f"\nQ: {ans['question']}")
        print(f"A: {ans['answer']}")
        print(f"Why: {ans['reasoning']}")


# ------------------------------------------------------------
# 5. Save responses
# ------------------------------------------------------------

with open("simulated_survey_responses.json", "w") as f:
    json.dump(all_responses, f, indent=2)

print("\n✅ Saved to simulated_survey_responses.json")

Total questions: 15
Simulating responses for Emma (1/12)...
Simulating responses for Vivian (2/12)...
Simulating responses for Adriana (3/12)...
Simulating responses for Margherita (4/12)...
Simulating responses for Daliana (5/12)...
Simulating responses for Christopher (6/12)...
Simulating responses for Zachary (7/12)...
Simulating responses for Athina (8/12)...
Simulating responses for Daniel (9/12)...
Simulating responses for Alina (10/12)...
Simulating responses for Kevin (11/12)...
Simulating responses for Leon (12/12)...

✅ Simulation complete.

PERSONA: Emma

Q: How interested are you in using a financial product like FlexiSave Kids Plan for your child?
A: Somewhat interested
Why: I like the idea of structured savings and teaching my child about money early, but I’m cautious about new financial products and need to ensure it fits my busy schedule.

Q: Which feature of the FlexiSave Kids Plan appeals to you the most?
A: Goal-based savings for children's future needs
Why: Having c

In [29]:
# ============================================================
# STEP 10A: Inspect response structure
# ============================================================

print("Number of response sets:", len(all_responses))

if len(all_responses) > 0:
    print("\nTop-level keys:")
    print(all_responses[0].keys())

    print("\nFirst response item:")
    print(all_responses[0]["responses"][0])

Number of response sets: 12

Top-level keys:
dict_keys(['persona_name', 'responses'])

First response item:
{'question': 'How interested are you in using a financial product like FlexiSave Kids Plan for your child?', 'answer': 'Somewhat interested', 'reasoning': 'I like the idea of structured savings and teaching my child about money early, but I’m cautious about new financial products and need to ensure it fits my busy schedule.'}


In [30]:
# ============================================================
# STEP 10B: Aggregate insights safely
# ============================================================

import json

if len(all_responses) == 0:
    raise ValueError("No responses found. Run Step 9 first.")

flat_data = []

for persona in all_responses:
    for r in persona.get("responses", []):
        flat_data.append({
            "persona": persona.get("persona_name", ""),
            "segment": persona.get("segment", ""),
            "question": r.get("question", ""),
            "answer": r.get("answer", ""),
            # safely handle different possible names
            "reasoning": (
                r.get("reasoning")
                or r.get("reason")
                or r.get("explanation")
                or r.get("rationale")
                or ""
            )
        })

print("Total flattened responses:", len(flat_data))

print("\nSample flattened response:")
print(flat_data[0])

Total flattened responses: 180

Sample flattened response:
{'persona': 'Emma', 'segment': '', 'question': 'How interested are you in using a financial product like FlexiSave Kids Plan for your child?', 'answer': 'Somewhat interested', 'reasoning': 'I like the idea of structured savings and teaching my child about money early, but I’m cautious about new financial products and need to ensure it fits my busy schedule.'}


In [31]:
# ============================================================
# STEP 10C: Extract strategic insights
# ============================================================

def extract_insights(flat_data):
    prompt = f"""
You are a senior product strategist.

We tested this financial product using simulated personas:

{financial_product}

Here are the simulated survey responses:
{json.dumps(flat_data[:250], indent=2)}

Identify:
1. Top 5 recurring needs
2. Top 5 recurring pain points
3. Top 5 objections
4. Most interested persona segments
5. Least interested persona segments
6. Key rejection reasons
7. Key features that drive interest
8. 3 product improvements
9. 1 clear product positioning statement

Return valid JSON only using this structure:

{{
  "needs": [],
  "pain_points": [],
  "objections": [],
  "most_interested_segments": [],
  "least_interested_segments": [],
  "rejection_reasons": [],
  "key_features": [],
  "product_improvements": [],
  "positioning": ""
}}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        response_format={"type": "json_object"},
        messages=[
            {
                "role": "system",
                "content": "You analyze product research responses and return valid JSON only."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.3
    )

    return json.loads(response.choices[0].message.content)


insights = extract_insights(flat_data)

print("✅ Insights extracted.")

✅ Insights extracted.


In [32]:
# ============================================================
# STEP 10D: Display insights
# ============================================================

for key, value in insights.items():
    print("\n" + "="*60)
    print(key.upper())
    print("="*60)

    if isinstance(value, list):
        for v in value:
            print("-", v)
    else:
        print(value)


NEEDS
- Goal-based savings for children's future needs
- Monthly automatic contributions
- Customization of savings goals
- Educational features for children
- Transparency regarding fees and investment risks

PAIN_POINTS
- Concerns about hidden fees
- Complexity of investment features
- Time constraints for managing the product
- Uncertainty about the effectiveness of educational nudges
- Difficulty in maintaining consistent contributions

OBJECTIONS
- Skepticism about the safety and effectiveness of micro-investing
- Caution regarding new financial products
- Need for proof of benefits before recommending
- Concerns about managing multiple financial goals
- Desire for clear and simple terms

MOST_INTERESTED_SEGMENTS
- Emma
- Vivian
- Adriana
- Margherita
- Daliana

LEAST_INTERESTED_SEGMENTS
- Alina
- Kevin
- Leon

REJECTION_REASONS
- Need for transparency in fees and terms
- Concerns about hidden costs
- Skepticism about the product's complexity

KEY_FEATURES
- Goal-based savings
- 